# SheCraft retrieval method comparison

Compare keyword search (TF-IDF), E5 dense retrieval, and hybrid retrieval with equal-weight reciprocal rank fusion (RRF). All methods use the same fixed-size chunks and draft questions. A hit requires the expected PDF and page.

The extended comparison includes dense Qdrant, dense and hybrid with cross-encoder reranking, and Gemini multi-query dense retrieval. Qdrant runs in memory on the same corpus. Gemini uses cached paraphrases by default. The notebook reports methods skipped because a prerequisite is missing.


## 1. Set up the local notebook kernel

In [1]:
import os
import importlib
import json
import sys
from pathlib import Path
from shutil import which
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (candidate for folder in (cwd, *cwd.parents)
     for candidate in (folder, folder / "ai-service")
     if (candidate / "src" / "rag" / "doc_rag").is_dir()
     and (candidate / "rag_documents").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from the SheCraft Jewelry workspace.")
SRC_DIR = PROJECT_ROOT / "src" / "rag"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    import torch
    from sentence_transformers import SentenceTransformer
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Retrieval comparison needs {error.name}. Install the model dependencies in the SheCraft notebook kernel, then rerun."
    ) from error

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Embedding device:", DEVICE)

C:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\tmp\shecraft-notebooks\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding device: cpu


## 2. Build the shared corpus and load the draft questions

In [2]:
CHUNKING_STRATEGY = "fixed_size"
CHUNKING_OPTIONS = {"chunk_size": 800, "overlap": 100}
MODEL_NAME = "intfloat/e5-small-v2"
BATCH_SIZE = 8
TOP_K = 5
CANDIDATE_K = 30
RRF_CONSTANT = 60
RERANKER_NAME = "cross-encoder/ms-marco-MiniLM-L2-v2"
ALLOW_RERANKER_DOWNLOAD = True
GENERATE_GEMINI_VARIANTS = False
GEMINI_VARIANT_COUNT = 3

extraction_module = importlib.import_module("doc_rag.2_extraction")
chunking_module = importlib.import_module("doc_rag.4_chunking")
embedding_module = importlib.import_module("doc_rag.5_embeddings")
extract_pdf = extraction_module.extract_pdf
chunk_extracted_documents = chunking_module.chunk_extracted_documents

DOCUMENTS_DIR = PROJECT_ROOT / "rag_documents"
TITLE_ONLY_PDFS = {"birthstone_guide.pdf"}
PDF_FILES = sorted(path for path in DOCUMENTS_DIR.rglob("*.pdf") if path.name not in TITLE_ONLY_PDFS)
CASES_FILE = PROJECT_ROOT / "data" / "rag" / "experiments" / "chunking_evaluation" / "document_chunking_cases.json"
with CASES_FILE.open(encoding="utf-8") as file:
    evaluation_payload = json.load(file)
evaluation_cases = evaluation_payload["cases"]
if evaluation_payload.get("status") != "draft":
    raise ValueError("Notebook 5 expects a provisional draft evaluation set.")

expected_files = {case["expected_file"] for case in evaluation_cases}
found_files = {path.name for path in PDF_FILES}
if len(found_files) != len(PDF_FILES):
    raise ValueError("PDF names must be unique for page-based scoring.")
if len({case["id"] for case in evaluation_cases}) != len(evaluation_cases):
    raise ValueError("Evaluation case IDs must be unique.")
if expected_files != found_files:
    raise ValueError(f"Evaluation/document coverage mismatch. Missing PDFs: {sorted(expected_files - found_files)}; PDFs without cases: {sorted(found_files - expected_files)}")

extracted_documents = [extract_pdf(path, use_ocr_fallback=which("tesseract") is not None) for path in PDF_FILES]
chunks = chunk_extracted_documents(
    extracted_documents, strategy=CHUNKING_STRATEGY, **CHUNKING_OPTIONS
)
available_pages = set(zip(chunks["file_name"], chunks["page_number"]))
missing_labels = [
    (case["id"], case["expected_file"], page)
    for case in evaluation_cases
    for page in case["expected_pages"]
    if (case["expected_file"], page) not in available_pages
]
if not evaluation_cases or missing_labels:
    raise ValueError(f"Missing evaluation cases or labeled pages: {missing_labels}")
print(f"Fixed corpus: {len(chunks)} chunks, {len(expected_files)} PDFs")
print("Draft evaluation questions:", len(evaluation_cases))


Fixed corpus: 301 chunks, 39 PDFs
Draft evaluation questions: 39


## 3. Build both indexes before timing

In [3]:
tfidf = TfidfVectorizer(
    lowercase=True, strip_accents=None, token_pattern=r"(?u)\b\w\w+\b",
    ngram_range=(1, 2), sublinear_tf=True,
)
tfidf_index = tfidf.fit_transform(chunks["text"].tolist())

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
passages = embedding_module.prepare_texts(
    chunks["text"].tolist(), model_name=MODEL_NAME, text_type="passage"
)
dense_index = embedding_module.embed_texts(passages, model, batch_size=BATCH_SIZE)
print("Indexes ready:", len(chunks), "chunks per method")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  18%|█▊        | 35/199 [00:00<00:00, 337.47it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1062.50it/s]

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Batches:   3%|▎         | 1/38 [00:01<00:43,  1.17s/it]

Batches:   5%|▌         | 2/38 [00:01<00:31,  1.15it/s]

... (34 outputs left out)

Batches:  97%|█████████▋| 37/38 [00:25<00:00,  3.73it/s]

Batches: 100%|██████████| 38/38 [00:25<00:00,  4.43it/s]

Batches: 100%|██████████| 38/38 [00:25<00:00,  1.46it/s]

Indexes ready: 301 chunks per method


## 4. Rank chunks for each question

In [4]:
def tfidf_candidates(query):
    vector = tfidf.transform([query])
    scores = (vector @ tfidf_index.T).toarray()[0]
    positive = np.flatnonzero(scores > 0)
    positions = positive[np.argsort(-scores[positive], kind="stable")][:CANDIDATE_K]
    return [(int(pos), float(scores[pos])) for pos in positions]


def dense_candidates(query):
    vector = embedding_module.embed_queries([query], model=model, model_name=MODEL_NAME)
    scores, positions = embedding_module.cosine_search(vector, dense_index, top_k=CANDIDATE_K)
    return [(int(pos), float(score)) for pos, score in zip(positions[0], scores[0])]


def fuse_rankings(rankings, weight):
    fused = {}
    for ranking in rankings:
        for rank, (position, _) in enumerate(ranking, start=1):
            fused[position] = fused.get(position, 0.0) + weight / (RRF_CONSTANT + rank)
    return sorted(fused.items(), key=lambda item: -item[1])


def hybrid_candidates(query):
    return fuse_rankings((dense_candidates(query), tfidf_candidates(query)), weight=0.5)


def rerank(query, candidates):
    positions = [position for position, _ in candidates[:CANDIDATE_K]]
    pairs = [(query, chunks.iloc[position]["text"]) for position in positions]
    scores = reranker.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=False)
    return sorted(zip(positions, map(float, scores)), key=lambda item: -item[1])


def qdrant_candidates(query):
    vector = embedding_module.embed_queries([query], model=model, model_name=MODEL_NAME)
    hits = qdrant.query_points(
        collection_name=collection,
        query=vector[0].tolist(),
        limit=CANDIDATE_K,
        with_payload=False,
    ).points
    return [(int(hit.id), float(hit.score)) for hit in hits]


def multi_query_candidates(query):
    return fuse_rankings(
        (dense_candidates(variant) for variant in query_variants[query]), weight=1.0
    )


methods = {
    "tfidf": tfidf_candidates,
    "dense": dense_candidates,
    "hybrid_rrf": hybrid_candidates,
}
skipped = {}
try:
    from qdrant_client import QdrantClient, models
except ImportError as error:
    skipped["dense_qdrant"] = f"Install qdrant-client in this kernel: {error}"
else:
    qdrant = QdrantClient(":memory:")
    collection = "shecraft_notebook5"
    qdrant.create_collection(
        collection,
        vectors_config=models.VectorParams(size=dense_index.shape[1], distance=models.Distance.COSINE),
    )
    qdrant.upload_collection(
        collection, vectors=dense_index, ids=list(range(len(chunks))), wait=True,
    )
    methods["dense_qdrant"] = qdrant_candidates

try:
    from sentence_transformers import CrossEncoder
    reranker = CrossEncoder(
        RERANKER_NAME, device=DEVICE, local_files_only=not ALLOW_RERANKER_DOWNLOAD,
    )
except (ImportError, OSError, RuntimeError, ValueError) as error:
    reason = f"Could not load {RERANKER_NAME}: {error}"
    skipped.update(dense_reranked=reason, hybrid_reranked=reason)
else:
    methods["dense_reranked"] = lambda query: rerank(query, dense_candidates(query))
    methods["hybrid_reranked"] = lambda query: rerank(query, hybrid_candidates(query))

variants_file = (
    PROJECT_ROOT / "data" / "rag" / "experiments"
    / "retrieval_method_comparison" / "gemini_query_variants.json"
)
queries = [case["query"] for case in evaluation_cases]
if GENERATE_GEMINI_VARIANTS:
    from google import genai

    # Supply these in the kernel environment before requesting external paraphrases.
    gemini_model = os.environ["GEMINI_MODEL"]
    gemini = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    retrieval_module = importlib.import_module("doc_rag.7_retrieval")
    query_variants = retrieval_module.get_or_create_gemini_query_variants(
        queries, gemini, cache_file=variants_file,
        gemini_model=gemini_model, variant_count=GEMINI_VARIANT_COUNT,
    )
elif variants_file.is_file():
    variants_payload = json.loads(variants_file.read_text(encoding="utf-8"))
    gemini_model = variants_payload.get("gemini_model")
    cached_variants = variants_payload["variants"]
    query_variants = {
        query: list(dict.fromkeys([query, *cached_variants.get(query, [])])) for query in queries
    }
else:
    gemini_model, query_variants = None, {}

if all(len(query_variants.get(query, [])) > 1 for query in queries):
    methods["multi_query"] = multi_query_candidates
else:
    skipped["multi_query"] = "A complete Gemini paraphrase cache is required."
print("Methods:", ", ".join(methods))
for name, reason in skipped.items():
    print(f"Skipped {name}: {reason}")

method_order = list(methods)
for retrieve in methods.values():
    retrieve(evaluation_cases[0]["query"])

case_rows = []
for case_index, case in enumerate(evaluation_cases):
    expected_pages = set(case["expected_pages"])
    # Rotate the order to reduce timing bias.
    offset = case_index % len(method_order)
    for name in method_order[offset:] + method_order[:offset]:
        started = perf_counter()
        ranked = methods[name](case["query"])[:TOP_K]
        latency_ms = (perf_counter() - started) * 1000
        hits = []
        for rank, (position, score) in enumerate(ranked, start=1):
            chunk = chunks.iloc[position]
            page = int(chunk["page_number"])
            relevant = chunk["file_name"] == case["expected_file"] and page in expected_pages
            hits.append({
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "file_name": chunk["file_name"],
                "page_number": page,
                "score": score,
                "relevant": bool(relevant),
            })
        case_rows.append({
            "method": name,
            "case_id": case["id"],
            "query": case["query"],
            "expected_file": case["expected_file"],
            "expected_pages": case["expected_pages"],
            "first_relevant_rank": next((hit["rank"] for hit in hits if hit["relevant"]), None),
            "latency_ms": latency_ms,
            "top_results": hits,
        })
print(f"Scored {len(case_rows)} method/question pairs")


Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 41/41 [00:00<00:00, 3014.68it/s]

Methods: tfidf, dense, hybrid_rrf, dense_qdrant, dense_reranked, hybrid_reranked
Skipped multi_query: A complete Gemini paraphrase cache is required.


Scored 234 method/question pairs


## 5. Compare quality and latency

Hit rate measures how often an expected page appears in the first K results. MRR@5 rewards earlier hits. Latency measures one warm query, including encoding, ranking, and fusion as applicable. Index building and model loading happen before timing.

Multi-query timing uses cached paraphrases and covers the retrieval and fusion steps.


In [5]:
case_results = pd.DataFrame(case_rows)
metric_rows = []
for name, group in case_results.groupby("method", sort=False):
    ranks = group["first_relevant_rank"].astype(float)
    metric_rows.append({
        "method": name,
        "cases": len(group),
        "hit_rate_at_1": float(ranks.le(1).mean()),
        "hit_rate_at_3": float(ranks.le(3).mean()),
        "hit_rate_at_5": float(ranks.le(5).mean()),
        "mrr_at_5": float((1 / ranks).fillna(0).mean()),
        "median_latency_ms": float(group["latency_ms"].median()),
        "p95_latency_ms": float(group["latency_ms"].quantile(0.95)),
    })

metrics = pd.DataFrame(metric_rows).sort_values(
    ["hit_rate_at_5", "mrr_at_5", "hit_rate_at_1"], ascending=False
)
display(metrics)
misses = case_results.loc[
    case_results["first_relevant_rank"].isna(),
    ["method", "case_id", "query", "expected_file", "expected_pages"],
]
print("Top-five misses:", len(misses))
if not misses.empty:
    display(misses)
display(case_results.pivot(index="case_id", columns="method", values="first_relevant_rank"))


,method,cases,hit_rate_at_1,hit_rate_at_3,hit_rate_at_5,mrr_at_5,median_latency_ms,p95_latency_ms
2,hybrid_rrf,39,0.923077,1.000000,1.0,0.957265,40.2261,119.16549
1,dense,39,0.923077,1.000000,1.0,0.952991,38.5581,74.18211
3,dense_qdrant,39,0.923077,1.000000,1.0,0.952991,38.6819,70.11069
4,dense_reranked,39,0.871795,1.000000,1.0,0.935897,500.7913,730.78025
5,hybrid_reranked,39,0.871795,1.000000,1.0,0.935897,515.4133,858.55887
0,tfidf,39,0.615385,0.923077,1.0,0.770085,3.2764,4.33166


Top-five misses: 0


method,dense,dense_qdrant,dense_reranked,hybrid_reranked,hybrid_rrf,tfidf
case_id,,,,,,
bench_pave,1,1,1,1,1,1
buying_gemstones,1,1,1,1,1,1
chain_durability,1,1,2,2,1,3
everyday_ring_metal,1,1,2,2,1,2
gem_alexandrite,1,1,1,1,1,1
gem_amber,1,1,1,1,1,2
gem_amethyst,1,1,1,1,1,5
gem_ametrine,1,1,1,1,1,1
gem_aquamarine,1,1,1,1,1,3


## 6. Save the draft results

In [6]:
results_file = (
    PROJECT_ROOT / "data" / "rag" / "experiments" / "retrieval_method_comparison" / "results.json"
)
payload = {
    "schema_version": 1,
    "status": "draft",
    "evaluation_case_file": CASES_FILE.relative_to(PROJECT_ROOT).as_posix(),
    "relevance": "expected PDF and page",
    "chunking_strategy": CHUNKING_STRATEGY,
    "chunking_options": CHUNKING_OPTIONS,
    "chunk_count": len(chunks),
    "embedding_model": MODEL_NAME,
    "device": DEVICE,
    "top_k": TOP_K,
    "methods_run": list(methods),
    "methods_skipped": skipped,
    "qdrant_mode": "in_memory" if "dense_qdrant" in methods else None,
    "reranker_model": RERANKER_NAME if "dense_reranked" in methods else None,
    "gemini_model": gemini_model if "multi_query" in methods else None,
    "rrf_candidate_k": CANDIDATE_K,
    "rrf_constant": RRF_CONSTANT,
    "rrf_weights": {"dense": 0.5, "tfidf": 0.5},
    "metrics": metrics.to_dict(orient="records"),
    "case_results": case_rows,
}
results_file.parent.mkdir(parents=True, exist_ok=True)
results_file.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Saved draft retrieval comparison:", results_file)


Saved draft retrieval comparison: C:\Users\USP\OneDrive\Desktop\Fall 2026\Large Language Models\SheCraft\Jewelry\ai-service\data\rag\experiments\retrieval_method_comparison\results.json
